In [ ]:
import os

os.environ.setdefault("HF_HOME", "/nfsd/lttm4/tesisti/shahrampour/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "/nfsd/lttm4/tesisti/shahrampour/hf_transformers")

for k in ["HF_HOME","HF_DATASETS_CACHE","TRANSFORMERS_CACHE"]:
    os.makedirs(os.environ[k], exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE:", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])


## 1) Imports

In [ ]:
import numpy as np
import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil

from peft import LoraConfig, get_peft_model, PeftConfig
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This run must use GPU on the cluster.")

device = torch.device("cuda")
print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
RUN_NAME = "cifar100_5step_rankext_clean_n4"

BASE_OUT = os.path.join("outputs", RUN_NAME)

RESULTS_DIR = os.path.join(BASE_OUT, "results")
PLOTS_DIR = os.path.join(BASE_OUT, "plots")
TABLES_DIR = os.path.join(BASE_OUT, "tables")
METRICS_DIR = os.path.join(BASE_OUT, "metrics")

for d in [RESULTS_DIR, PLOTS_DIR, TABLES_DIR, METRICS_DIR]:
    os.makedirs(d, exist_ok=True)

STEP1_OUT = os.path.join(BASE_OUT, "step1_scratch_train")
STEP1_FINAL_OUT = os.path.join(BASE_OUT, "step1_scratch_final")

RANKEXT_DIR = os.path.join(BASE_OUT, "rankext_adapters")
os.makedirs(RANKEXT_DIR, exist_ok=True)

FT_OUT = os.path.join(BASE_OUT, "full_finetune")
JOINT_OUT = os.path.join(BASE_OUT, "joint_upper_bound")

## 2) Load CIFAR-100 (fine labels)

In [ ]:
from datasets import Image

dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")

dataset = dataset.cast_column("img", Image())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("example keys:", dataset["train"][0].keys())
print("first 10 classes:", class_names[:10])

In [ ]:
def make_custom_eval_dataset(class_ids, remap_labels=True):
    test_ds = filter_dataset_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        test_ds = test_ds.map(remap)
    else:
        label2new = None
        new2orig = None

    test_ds.set_transform(preprocess_val)
    return test_ds, label2new, new2orig

## 3) Define incremental class splits (2/5/10 steps)

In [ ]:
num_steps = 5  

assert num_classes % num_steps == 0
classes_per_step = num_classes // num_steps

class_splits = [
    list(range(s * classes_per_step, (s + 1) * classes_per_step))
    for s in range(num_steps)
]

print("classes per step:", classes_per_step)
print("split sizes:", [len(x) for x in class_splits])
print("step0 sample:", class_splits[0][:10], "...", class_splits[0][-3:])

In [ ]:
first_step_classes = class_splits[0]
later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(class_splits[s])
all_classes = list(range(num_classes))

print("first step classes:", len(first_step_classes))
print("later step classes:", len(later_step_classes))
print("all classes:", len(all_classes))

## 4) Model + preprocessing

In [ ]:
# Requested model
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint, use_fast=True)

from torchvision import transforms
from PIL import Image
import numpy as np
import torch

size = image_processor.size
if isinstance(size, dict):
    H = size.get("height", 224)
    W = size.get("width", 224)
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = x.astype(np.uint8)
        arr = np.squeeze(arr)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if not (arr.ndim == 3 and arr.shape[-1] in (1, 3)):
            raise TypeError(f"Unexpected image array shape after squeeze: {arr.shape}")

        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean().item()}

## 5) Build per-step datasets (step / cumulative / full)

In [ ]:
def classes_for_step(step_idx: int) -> list[int]:
    return class_splits[step_idx]

def classes_for_cumulative(step_idx: int) -> list[int]:
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def filter_by_classes(ds, class_ids):
    class_set = set(class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda x: int(x["label"]) in class_set)

def filter_dataset_by_classes(ds, class_ids):
    class_ids = set(int(c) for c in class_ids)
    return ds.filter(lambda ex: int(ex["label"]) in class_ids)

def make_step_datasets(step_idx: int, split_type: str = "new_only", remap_labels: bool = False):
    """
    split_type:
      - 'new_only'   : only classes of this step
      - 'cumulative' : union of classes up to this step
      - 'full'       : all classes (100)
    """
    if split_type == "full":
        class_ids = list(range(num_classes))
    elif split_type == "cumulative":
        class_ids = classes_for_cumulative(step_idx)
    elif split_type == "new_only":
        class_ids = classes_for_step(step_idx)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")

    train_ds = filter_by_classes(dataset["train"], class_ids)
    test_ds  = filter_by_classes(dataset["test"], class_ids)

    if remap_labels:
        label2new = {orig: i for i, orig in enumerate(sorted(class_ids))}
        new2orig = {v: k for k, v in label2new.items()}

        def remap(ex):
            ex["label"] = label2new[int(ex["label"])]
            return ex

        train_ds = train_ds.map(remap)
        test_ds  = test_ds.map(remap)
    else:
        label2new = {c: c for c in class_ids}
        new2orig = {c: c for c in class_ids}

    train_ds = train_ds.with_transform(preprocess_train)
    test_ds = test_ds.with_transform(preprocess_val)

    print(f"[Dataset] Step {step_idx+1} | split_type={split_type}")
    print(f"[Dataset] Classes: {class_ids[:5]} ... {class_ids[-5:]}")
    print(f"[Dataset] Train size: {len(train_ds)} | Test size: {len(test_ds)}")

    return train_ds, test_ds, label2new, new2orig, class_ids

eval_all = dataset["test"].with_transform(preprocess_val)

print("eval_all size:", len(eval_all))

## 6) Training recipes (reasonable settings)

In [ ]:
set_seed(42)

SCRATCH_EPOCHS = 10
RANKEXT_EPOCHS = 2
FT_EPOCHS      = 4
JOINT_EPOCHS   = 10

BATCH_SCRATCH = 8
ACCUM_SCRATCH = 2

BATCH_LORA    = 16
ACCUM_LORA    = 1

BATCH_FT      = 8
ACCUM_FT      = 2

LR_SCRATCH = 5e-5
LR_RANKEXT = 1.5e-5
LR_FT      = 3e-5
LR_JOINT   = 5e-5

WEIGHT_DECAY = 0.05
WEIGHT_DECAY_LORA = 0.01

WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# RankExt setup:
# CIFAR step 1 is trained as full scratch base.
# LoRA starts from CIFAR step 2.
RANKEXT_NEW_RANK_PER_STEP = 8
RANKEXT_ALPHA_PER_STEP = 16
TARGET_MODULES = ["query", "value"]

FREEZE_OLD_LORA_RANKS = True
FREEZE_OLD_CLASSIFIER_ROWS = True

results = []

In [ ]:
run_config = {
    "run_name": RUN_NAME,
    "model_checkpoint": model_checkpoint,
    "init_mode": "scratch_step1_then_rankext",
    "num_steps": num_steps,
    "classes_per_step": classes_per_step,
    "scratch_epochs": SCRATCH_EPOCHS,
    "rankext_epochs": RANKEXT_EPOCHS,
    "rankext_new_rank_per_step": RANKEXT_NEW_RANK_PER_STEP,
    "target_modules": TARGET_MODULES,
    "freeze_old_lora_ranks": FREEZE_OLD_LORA_RANKS,
    "freeze_old_classifier_rows": FREEZE_OLD_CLASSIFIER_ROWS,
    "note": (
        "RankExt uses one growing LoRA adapter. "
        "At each new step, the previous A/B blocks are copied into the larger-rank adapter "
        "and frozen; only the newly added rank block is trained."
    ),
}

with open(os.path.join(RESULTS_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(json.dumps(run_config, indent=2))

In [ ]:
def lambda_tag(x):
    s = f"{x:.0e}" if x < 1e-2 and x > 0 else str(x)
    return s.replace("-", "m").replace(".", "p")

## 7) Step 1: train full ViT from scratch on step 0 classes

In [ ]:
import os, shutil, json

# --- clean old step1 outputs so stale checkpoints cannot survive ---
if os.path.exists(STEP1_OUT):
    shutil.rmtree(STEP1_OUT)
if os.path.exists(STEP1_FINAL_OUT):
    shutil.rmtree(STEP1_FINAL_OUT)

os.makedirs(STEP1_OUT, exist_ok=True)
os.makedirs(STEP1_FINAL_OUT, exist_ok=True)

step1_idx = 0
train_step1, test_step1, label2new_1, new2orig_1, class_ids_1 = make_step_datasets(
    step1_idx, split_type="new_only", remap_labels=False
)

print("Step1 original classes:", class_ids_1[:5], "...", class_ids_1[-5:])
print(
    "Step1 label range:",
    min(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
    max(int(train_step1[i]["label"]) for i in range(min(200, len(train_step1)))),
)

config_step1 = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

model_step1 = AutoModelForImageClassification.from_config(config_step1)

print("Before training - Step1 model num_labels:", model_step1.config.num_labels)
print("Before training - Step1 classifier out_features:", model_step1.classifier.out_features)
assert model_step1.config.num_labels == num_classes
assert model_step1.classifier.out_features == num_classes

args_step1 = TrainingArguments(
    output_dir=STEP1_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=SCRATCH_EPOCHS,
    learning_rate=LR_SCRATCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_SCRATCH,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_SCRATCH,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    max_grad_norm=1.0,
)

trainer_step1 = Trainer(
    model=model_step1,
    args=args_step1,
    train_dataset=train_step1,
    eval_dataset=test_step1,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback
trainer_step1.remove_callback(NotebookProgressCallback)

trainer_step1.train()

eval_step1 = trainer_step1.evaluate()
print({"eval_step1": eval_step1})

print("About to save B0 final model to:", STEP1_FINAL_OUT)
print("Trainer model type:", type(trainer_step1.model))
print("Trainer model class:", trainer_step1.model.__class__.__name__)

trainer_step1.model.save_pretrained(STEP1_FINAL_OUT, safe_serialization=False)
image_processor.save_pretrained(STEP1_FINAL_OUT)

print("Saved STEP1_FINAL_OUT to:", STEP1_FINAL_OUT)
print("STEP1_FINAL_OUT exists:", os.path.exists(STEP1_FINAL_OUT))
print("Files in STEP1_FINAL_OUT:", os.listdir(STEP1_FINAL_OUT) if os.path.exists(STEP1_FINAL_OUT) else "MISSING")

cfg_path = os.path.join(STEP1_FINAL_OUT, "config.json")
print("config exists:", os.path.exists(cfg_path))

if os.path.exists(cfg_path):
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    print("model_type:", cfg.get("model_type"))
    print("architectures:", cfg.get("architectures"))
    print("num_labels:", cfg.get("num_labels"))

reloaded_step1 = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)
print("Reloaded STEP1 num_labels:", reloaded_step1.config.num_labels)
print("Reloaded STEP1 classifier out_features:", reloaded_step1.classifier.out_features)

assert reloaded_step1.config.num_labels == num_classes
assert reloaded_step1.classifier.out_features == num_classes

In [ ]:
print("Init mode:", run_config["init_mode"])
assert run_config["init_mode"] == "scratch"

In [ ]:
step1_log_df = pd.DataFrame(trainer_step1.state.log_history)
step1_log_df.to_csv(os.path.join(TABLES_DIR, "step1_log_history.csv"), index=False)

step1_metrics = {
    "experiment": "step1_scratch",
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "step1_metrics.json"), "w") as f:
    json.dump(step1_metrics, f, indent=2)

results.append({
    "experiment": "step1_scratch",
    "method": "step1_scratch",
    "step": 1,
    "eval_type": "current_step",
    "eval_accuracy": float(eval_step1.get("eval_accuracy", np.nan)),
    "eval_loss": float(eval_step1.get("eval_loss", np.nan)),
})

step1_log_df.tail()

In [ ]:
if "loss" in step1_log_df.columns:
    train_loss_df = step1_log_df[step1_log_df["loss"].notna()]
    if not train_loss_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(train_loss_df["step"], train_loss_df["loss"])
        plt.xlabel("Step")
        plt.ylabel("Train Loss")
        plt.title("Step 1 Train Loss")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_train_loss.png"), dpi=200)
        plt.show()

if "eval_accuracy" in step1_log_df.columns:
    eval_df = step1_log_df[step1_log_df["eval_accuracy"].notna()]
    if not eval_df.empty:
        plt.figure(figsize=(8,5))
        plt.plot(eval_df["epoch"], eval_df["eval_accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Eval Accuracy")
        plt.title("Step 1 Eval Accuracy")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "step1_eval_accuracy.png"), dpi=200)
        plt.show()

## 8) Step 2: LoRA only (freeze backbone) on top of Step 1 model

In [ ]:
first_step_classes = classes_for_step(0)

def make_eval_dataset(class_ids, name=None):
    test_ds = filter_by_classes(dataset["test"], class_ids)
    test_ds = test_ds.with_transform(preprocess_val)
    if name is not None:
        print(f"[Eval set] {name}: num_classes={len(class_ids)}, size={len(test_ds)}")
    return test_ds

In [ ]:
from safetensors.torch import load_file as safe_load_file
from peft import PeftModel

def load_adapter_state(adapter_dir):
    safe_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    bin_path = os.path.join(adapter_dir, "adapter_model.bin")

    if os.path.exists(safe_path):
        return safe_load_file(safe_path)
    if os.path.exists(bin_path):
        return torch.load(bin_path, map_location="cpu")

    raise FileNotFoundError(f"No adapter_model found in {adapter_dir}")


def find_state_key(param_name, state_dict):
    candidates = [
        param_name,
        param_name.replace(".default.", "."),
        param_name.replace(".modules_to_save.default", ""),
        param_name.replace(".original_module", ""),
    ]

    for k in candidates:
        if k in state_dict:
            return k

    # fallback: match by ending
    short = param_name.replace(".default.", ".")
    for k in state_dict.keys():
        if k.endswith(short.split("base_model.model.")[-1]):
            return k

    return None


def copy_previous_rankext_weights(model_lora, prev_adapter_dir, r_old):
    if prev_adapter_dir is None or r_old == 0:
        print("[RankExt] No previous adapter to copy.")
        return

    prev_sd = load_adapter_state(prev_adapter_dir)

    copied_A = 0
    copied_B = 0
    copied_classifier = 0

    with torch.no_grad():
        for name, p in model_lora.named_parameters():
            key = find_state_key(name, prev_sd)
            if key is None:
                continue

            old_p = prev_sd[key].to(device=p.device, dtype=p.dtype)

            if "lora_A" in name and "weight" in name:
                p[:r_old, :].copy_(old_p[:r_old, :])
                copied_A += 1

            elif "lora_B" in name and "weight" in name:
                p[:, :r_old].copy_(old_p[:, :r_old])
                copied_B += 1

            elif "classifier" in name and p.shape == old_p.shape:
                p.copy_(old_p)
                copied_classifier += 1

    print(f"[RankExt] copied A={copied_A}, B={copied_B}, classifier={copied_classifier}")


def freeze_old_lora_ranks(model_lora, r_old):
    if not FREEZE_OLD_LORA_RANKS or r_old == 0:
        print("[RankExt] No old LoRA ranks to freeze.")
        return

    def hook_A(grad):
        grad = grad.clone()
        grad[:r_old, :] = 0.0
        return grad

    def hook_B(grad):
        grad = grad.clone()
        grad[:, :r_old] = 0.0
        return grad

    nA, nB = 0, 0

    for _, module in model_lora.named_modules():
        if hasattr(module, "lora_A") and "default" in module.lora_A:
            module.lora_A["default"].weight.register_hook(hook_A)
            nA += 1

        if hasattr(module, "lora_B") and "default" in module.lora_B:
            module.lora_B["default"].weight.register_hook(hook_B)
            nB += 1

    print(f"[RankExt] frozen old ranks: A modules={nA}, B modules={nB}")


def get_trainable_classifier(model_lora):
    cw = model_lora.base_model.model.classifier

    if hasattr(cw, "modules_to_save"):
        return cw.modules_to_save["default"]

    return cw


def freeze_old_classifier_rows(model_lora, old_classes):
    if not FREEZE_OLD_CLASSIFIER_ROWS or len(old_classes) == 0:
        print("[RankExt] No classifier rows frozen.")
        return

    classifier = get_trainable_classifier(model_lora)

    def weight_hook(grad):
        grad = grad.clone()
        grad[old_classes, :] = 0.0
        return grad

    classifier.weight.register_hook(weight_hook)

    if classifier.bias is not None:
        def bias_hook(grad):
            grad = grad.clone()
            grad[old_classes] = 0.0
            return grad

        classifier.bias.register_hook(bias_hook)

    print("[RankExt] frozen classifier rows:", old_classes)


def print_rankext_sanity(model_lora, r_old, r_total):
    trainable = [n for n, p in model_lora.named_parameters() if p.requires_grad]

    print(f"[RankExt sanity] r_old={r_old}, r_total={r_total}")
    print("[RankExt sanity] first trainable params:")
    for n in trainable[:30]:
        print("  ", n)

    classifier = get_trainable_classifier(model_lora)
    print("Classifier weight requires_grad:", classifier.weight.requires_grad)
    print("Classifier bias requires_grad:", classifier.bias.requires_grad if classifier.bias is not None else None)

In [ ]:
rankext_results = []

prev_adapter_dir = None

for step_idx in range(1, num_steps):
    cifar_step = step_idx + 1

    r_old = (step_idx - 1) * RANKEXT_NEW_RANK_PER_STEP
    r_total = step_idx * RANKEXT_NEW_RANK_PER_STEP

    current_classes = classes_for_step(step_idx)

    old_classes = []
    for s in range(step_idx):
        old_classes.extend(classes_for_step(s))

    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx=step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print("\n" + "=" * 80)
    print(f"[RankExt] CIFAR step {cifar_step}")
    print(f"Current classes: {current_classes}")
    print(f"Old classes    : {old_classes}")
    print(f"Rank: old={r_old}, total={r_total}")
    print("Train size:", len(train_step), "| Test size:", len(test_step))
    print("=" * 80)

    base_model = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)

    lora_config = LoraConfig(
        r=r_total,
        lora_alpha=RANKEXT_ALPHA_PER_STEP,
        lora_dropout=0.05,
        bias="none",
        target_modules=TARGET_MODULES,
        modules_to_save=["classifier"],
    )

    model_rankext = get_peft_model(base_model, lora_config)

    copy_previous_rankext_weights(
        model_lora=model_rankext,
        prev_adapter_dir=prev_adapter_dir,
        r_old=r_old,
    )

    freeze_old_lora_ranks(model_rankext, r_old)
    freeze_old_classifier_rows(model_rankext, old_classes)
    print_rankext_sanity(model_rankext, r_old, r_total)

    step_out = os.path.join("outputs", RUN_NAME, f"rankext_step_{cifar_step}_r{r_total}")

    args_rankext = TrainingArguments(
        output_dir=step_out,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=RANKEXT_EPOCHS,
        learning_rate=LR_RANKEXT,
        weight_decay=WEIGHT_DECAY_LORA,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_LORA,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_LORA,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_rankext = Trainer(
        model=model_rankext,
        args=args_rankext,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_rankext.train()

    eval_current = trainer_rankext.evaluate(eval_dataset=test_step)

    eval_first = make_eval_dataset(classes_for_step(0))
    seen_now = classes_for_cumulative(step_idx)
    later_now = [c for c in seen_now if c not in classes_for_step(0)]

    eval_later = make_eval_dataset(later_now)
    eval_seen = make_eval_dataset(seen_now)

    eval_first_res = trainer_rankext.evaluate(eval_dataset=eval_first)
    eval_later_res = trainer_rankext.evaluate(eval_dataset=eval_later)
    eval_seen_res = trainer_rankext.evaluate(eval_dataset=eval_seen)

    for eval_name, eval_res in [
        ("current_step", eval_current),
        ("first_step", eval_first_res),
        ("later_steps", eval_later_res),
        ("all_seen", eval_seen_res),
    ]:
        rankext_results.append({
            "experiment": "rankext_step",
            "method": "rank_extension",
            "step": cifar_step,
            "rank_total": r_total,
            "eval_type": eval_name,
            **eval_res,
        })

    adapter_save_dir = os.path.join(RANKEXT_DIR, f"step_{cifar_step}_rank_{r_total}_adapter")
    trainer_rankext.model.save_pretrained(adapter_save_dir)
    prev_adapter_dir = adapter_save_dir

    print(f"[RankExt] Step {cifar_step} current:", eval_current)
    print(f"[RankExt] Step {cifar_step} first  :", eval_first_res)
    print(f"[RankExt] Step {cifar_step} later  :", eval_later_res)
    print(f"[RankExt] Step {cifar_step} seen   :", eval_seen_res)

    # Debug predictions on first-step classes
    dbg_batch = collate_fn([eval_first[i] for i in range(64)])
    dbg_batch = {k: v.to(device) for k, v in dbg_batch.items()}

    trainer_rankext.model.eval()
    with torch.no_grad():
        dbg_logits = trainer_rankext.model(pixel_values=dbg_batch["pixel_values"]).logits
        dbg_preds = dbg_logits.argmax(dim=1).detach().cpu().tolist()

    print("[DEBUG RankExt first-step pred unique]:", sorted(set(dbg_preds)))

## 9) Baseline: full fine-tune instead of LoRA (Step 2)

In [ ]:
ft_results = []

for s in range(2, num_steps + 1):
    stale_train_dir = f"outputs/{RUN_NAME}/step_{s}_ft"
    stale_final_dir = f"outputs/{RUN_NAME}/step_{s}_ft_final"
    if os.path.exists(stale_train_dir):
        shutil.rmtree(stale_train_dir)
    if os.path.exists(stale_final_dir):
        shutil.rmtree(stale_final_dir)

first_step_classes = classes_for_step(0)

def _label_range(ds, n=200):
    vals = [int(ds[i]["label"]) for i in range(min(n, len(ds)))]
    return min(vals), max(vals)

base_model_dir = STEP1_FINAL_OUT

for step_idx in range(1, num_steps):
    train_step, test_step, label2new, new2orig, class_ids = make_step_datasets(
        step_idx,
        split_type="new_only",
        remap_labels=False,
    )

    print(f"\n[FT] Step {step_idx+1}")
    print("Current step classes:", class_ids[:5], "...", class_ids[-5:])

    tr_min, tr_max = _label_range(train_step)
    te_min, te_max = _label_range(test_step)

    print("Train label range:", tr_min, tr_max)
    print("Test label range:", te_min, te_max)
    print("Num labels:", num_classes)

    print("Loaded base_model_dir:", base_model_dir)
    ft_model = AutoModelForImageClassification.from_pretrained(base_model_dir)

    print("Loaded FT model num_labels:", ft_model.config.num_labels)
    print("Loaded FT classifier out_features:", ft_model.classifier.out_features)

    assert ft_model.config.num_labels == num_classes, (
        f"Loaded FT model num_labels={ft_model.config.num_labels}, expected {num_classes}"
    )
    assert ft_model.classifier.out_features == num_classes, (
        f"Loaded FT classifier out_features={ft_model.classifier.out_features}, expected {num_classes}"
    )

    assert tr_min >= 0
    assert tr_max < ft_model.classifier.out_features, (
        f"Train labels out of range: max label {tr_max}, classifier out_features {ft_model.classifier.out_features}"
    )
    assert te_min >= 0
    assert te_max < ft_model.classifier.out_features, (
        f"Test labels out of range: max label {te_max}, classifier out_features {ft_model.classifier.out_features}"
    )

    step_ft_train_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft"

    args_ft = TrainingArguments(
        output_dir=step_ft_train_dir,
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        num_train_epochs=FT_EPOCHS,
        learning_rate=LR_FT,
        weight_decay=0.01,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=BATCH_FT,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=ACCUM_FT,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        report_to="none",
        max_grad_norm=1.0,
    )

    trainer_ft = Trainer(
        model=ft_model,
        args=args_ft,
        train_dataset=train_step,
        eval_dataset=test_step,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    trainer_ft.train()
    eval_current = trainer_ft.evaluate(test_step)

    pd.DataFrame(trainer_ft.state.log_history).to_csv(
        os.path.join(TABLES_DIR, f"step{step_idx+1}_ft_log_history.csv"),
        index=False
    )

    step_ft_dir = f"outputs/{RUN_NAME}/step_{step_idx+1}_ft_final"
    os.makedirs(step_ft_dir, exist_ok=True)

    print(f"[FT] Step {step_idx+1} saving model to {step_ft_dir}")
    trainer_ft.save_model(step_ft_dir)
    image_processor.save_pretrained(step_ft_dir)
    print(f"[FT] Step {step_idx+1} save finished")

    reloaded_ft = AutoModelForImageClassification.from_pretrained(step_ft_dir)
    print(f"[FT] Reload check step {step_idx+1} num_labels:", reloaded_ft.config.num_labels)
    print(f"[FT] Reload check step {step_idx+1} classifier out_features:", reloaded_ft.classifier.out_features)

    assert reloaded_ft.config.num_labels == num_classes, (
        f"Reloaded saved FT num_labels={reloaded_ft.config.num_labels}, expected {num_classes}"
    )
    assert reloaded_ft.classifier.out_features == num_classes, (
        f"Reloaded saved FT classifier out_features={reloaded_ft.classifier.out_features}, expected {num_classes}"
    )

    base_model_dir = step_ft_dir

    seen_classes_now = classes_for_cumulative(step_idx)
    eval_first = make_eval_dataset(first_step_classes, name=f"ft_step{step_idx+1}_first_step")

    later_seen_now = [c for c in seen_classes_now if c not in first_step_classes]
    eval_later = make_eval_dataset(later_seen_now, name=f"ft_step{step_idx+1}_later_seen") if len(later_seen_now) > 0 else None
    eval_seen = make_eval_dataset(seen_classes_now, name=f"ft_step{step_idx+1}_all_seen")

    metrics_first = trainer_ft.evaluate(eval_first)
    metrics_seen = trainer_ft.evaluate(eval_seen)

    if eval_later is not None:
        metrics_later = trainer_ft.evaluate(eval_later)
    else:
        metrics_later = {"eval_accuracy": np.nan, "eval_loss": np.nan}

    ft_results.extend([
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "current_step",
            "eval_accuracy": float(eval_current.get("eval_accuracy", np.nan)),
            "eval_loss": float(eval_current.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "first_step",
            "eval_accuracy": float(metrics_first.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_first.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "later_steps_seen_so_far",
            "eval_accuracy": float(metrics_later.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_later.get("eval_loss", np.nan)),
        },
        {
            "experiment": f"ft_step_{step_idx+1}",
            "method": "full_finetune",
            "step": step_idx + 1,
            "eval_type": "all_seen",
            "eval_accuracy": float(metrics_seen.get("eval_accuracy", np.nan)),
            "eval_loss": float(metrics_seen.get("eval_loss", np.nan)),
        },
    ])

print("\nFull FT continual training finished.")

In [ ]:
first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

final_ft_dir = f"outputs/{RUN_NAME}/step_{num_steps}_ft_final"
final_ft_model = AutoModelForImageClassification.from_pretrained(final_ft_dir)

args_ft_eval = TrainingArguments(
    output_dir=f"outputs/{RUN_NAME}/final_ft_eval",
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_ft_final = Trainer(
    model=final_ft_model,
    args=args_ft_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

ft_test_first = make_eval_dataset(first_step_classes)
ft_test_later = make_eval_dataset(later_step_classes)
ft_test_seen = make_eval_dataset(final_seen_classes)

print("Final FT num_labels:", final_ft_model.config.num_labels)
print("Final FT classifier out_features:", final_ft_model.classifier.out_features)
assert final_ft_model.classifier.out_features == num_classes

ft_final_first = trainer_ft_final.evaluate(ft_test_first)
ft_final_later = trainer_ft_final.evaluate(ft_test_later)
ft_final_seen = trainer_ft_final.evaluate(ft_test_seen)

print("FT final - first step:", ft_final_first)
print("FT final - later steps:", ft_final_later)
print("FT final - all seen:", ft_final_seen)

## 10) Upper bound: joint training (full dataset)

In [ ]:
train_joint, test_joint, label2new_J, new2orig_J, class_ids_J = make_step_datasets(
    step_idx=0, split_type="full", remap_labels=False
)

config_joint = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=num_classes,
    id2label={i: str(i) for i in range(num_classes)},
    label2id={str(i): i for i in range(num_classes)},
)

joint_model = AutoModelForImageClassification.from_config(config_joint)

args_joint = TrainingArguments(
    output_dir=JOINT_OUT,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    num_train_epochs=JOINT_EPOCHS,
    learning_rate=LR_JOINT,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=SCHED,
    per_device_train_batch_size=BATCH_FT,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM_FT,
    fp16=USE_FP16,
    dataloader_num_workers=4,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
)

trainer_joint = Trainer(
    model=joint_model,
    args=args_joint,
    train_dataset=train_joint,
    eval_dataset=test_joint,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer_joint.train()

eval_joint = trainer_joint.evaluate()
print({"eval_joint_full": eval_joint})

first_step_classes = classes_for_step(0)

later_step_classes = []
for s in range(1, num_steps):
    later_step_classes.extend(classes_for_step(s))

final_seen_classes = classes_for_cumulative(num_steps - 1)

joint_eval_first = make_eval_dataset(first_step_classes, name="joint_first")
joint_eval_later = make_eval_dataset(later_step_classes, name="joint_later")
joint_eval_seen  = make_eval_dataset(final_seen_classes, name="joint_seen")

joint_final_first = trainer_joint.evaluate(eval_dataset=joint_eval_first)
joint_final_later = trainer_joint.evaluate(eval_dataset=joint_eval_later)
joint_final_seen  = trainer_joint.evaluate(eval_dataset=joint_eval_seen)

print("Joint final - first_step:", joint_final_first)
print("Joint final - later_steps:", joint_final_later)
print("Joint final - all_seen:", joint_final_seen)

In [ ]:
joint_log_df = pd.DataFrame(trainer_joint.state.log_history)
joint_log_df.to_csv(os.path.join(TABLES_DIR, "joint_log_history.csv"), index=False)

joint_metrics = {
    "experiment": "joint_full",
    "eval_loss": float(eval_joint.get("eval_loss", np.nan)),
    "eval_accuracy": float(eval_joint.get("eval_accuracy", np.nan)),
}

with open(os.path.join(METRICS_DIR, "joint_metrics.json"), "w") as f:
    json.dump(joint_metrics, f, indent=2)

joint_log_df.tail()

In [ ]:
joint_test_first = make_eval_dataset(first_step_classes)
joint_test_later = make_eval_dataset(later_step_classes)
joint_test_all = make_eval_dataset(all_classes)

joint_final_first = trainer_joint.evaluate(joint_test_first)
joint_final_later = trainer_joint.evaluate(joint_test_later)
joint_final_all = trainer_joint.evaluate(joint_test_all)

print("Joint final - first step:", joint_final_first)
print("Joint final - later steps:", joint_final_later)
print("Joint final - all classes:", joint_final_all)

## 11) Compare results (step test vs full test)

In [ ]:
def grab_acc(d):
    return float(d["eval_accuracy"]) if "eval_accuracy" in d else np.nan

def grab_loss(d):
    return float(d["eval_loss"]) if "eval_loss" in d else np.nan

all_results = []

all_results.extend(results)
all_results.extend(rankext_results)
all_results.extend(ft_results)

# Final RankExt evaluation
final_rankext_adapter_dir = prev_adapter_dir
final_rankext_rank = (num_steps - 1) * RANKEXT_NEW_RANK_PER_STEP

base_final = AutoModelForImageClassification.from_pretrained(STEP1_FINAL_OUT)

final_lora_config = LoraConfig(
    r=final_rankext_rank,
    lora_alpha=RANKEXT_ALPHA_PER_STEP,
    lora_dropout=0.05,
    bias="none",
    target_modules=TARGET_MODULES,
    modules_to_save=["classifier"],
)

# load final adapter properly
final_rankext_model = PeftModel.from_pretrained(base_final, final_rankext_adapter_dir)
final_rankext_model = final_rankext_model.to(device)
final_rankext_model.eval()

args_final_eval = TrainingArguments(
    output_dir=os.path.join("outputs", RUN_NAME, "final_rankext_eval"),
    remove_unused_columns=False,
    report_to="none",
    fp16=USE_FP16,
    per_device_eval_batch_size=32,
)

trainer_rankext_final = Trainer(
    model=final_rankext_model,
    args=args_final_eval,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

rankext_final_first = trainer_rankext_final.evaluate(make_eval_dataset(first_step_classes))
rankext_final_later = trainer_rankext_final.evaluate(make_eval_dataset(later_step_classes))
rankext_final_seen = trainer_rankext_final.evaluate(make_eval_dataset(final_seen_classes))

for eval_name, eval_res in [
    ("first_step", rankext_final_first),
    ("later_steps", rankext_final_later),
    ("all_seen", rankext_final_seen),
]:
    all_results.append({
        "experiment": "rankext_final_eval",
        "method": "rank_extension",
        "step": num_steps,
        "rank_total": final_rankext_rank,
        "eval_type": eval_name,
        "eval_accuracy": grab_acc(eval_res),
        "eval_loss": grab_loss(eval_res),
    })

# Final FT
for eval_name, eval_res in [
    ("first_step", ft_final_first),
    ("later_steps", ft_final_later),
    ("all_seen", ft_final_seen),
]:
    all_results.append({
        "experiment": "ft_final_eval",
        "method": "full_finetune",
        "step": num_steps,
        "rank_total": np.nan,
        "eval_type": eval_name,
        "eval_accuracy": grab_acc(eval_res),
        "eval_loss": grab_loss(eval_res),
    })

# Joint
for eval_name, eval_res in [
    ("first_step", joint_final_first),
    ("later_steps", joint_final_later),
    ("all_seen", joint_final_all),
]:
    all_results.append({
        "experiment": "joint_final_eval",
        "method": "joint_upper_bound",
        "step": num_steps,
        "rank_total": np.nan,
        "eval_type": eval_name,
        "eval_accuracy": grab_acc(eval_res),
        "eval_loss": grab_loss(eval_res),
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(RESULTS_DIR, "all_results_rankext_clean.csv"), index=False)

results_df.head()

In [ ]:
results_df_clean = results_df.copy()

results_df_clean = results_df_clean.rename(columns={
    "eval_accuracy": "accuracy",
    "eval_loss": "loss",
    "eval_type": "eval_set",
})

def map_method(x):
    if x == "step1_scratch":
        return "B0 Scratch"
    if x == "rank_extension":
        return "RankExt"
    if x == "full_finetune":
        return "Full FT"
    if x == "joint_upper_bound":
        return "Joint"
    return x

results_df_clean["method"] = results_df_clean["method"].apply(map_method)

def map_eval_stage(exp_name):
    if "final_eval" in str(exp_name):
        return "final"
    if exp_name == "step1_scratch":
        return "step1"
    return "step"

results_df_clean["eval_stage"] = results_df_clean["experiment"].apply(map_eval_stage)

def adapter_name(row):
    if row["method"] == "B0 Scratch":
        return "B0"
    if row["method"] == "RankExt":
        r = row.get("rank_total", np.nan)
        return f"RankExt-r{int(r)}" if pd.notna(r) else "RankExt"
    return row["method"]

results_df_clean["adapter_name"] = results_df_clean.apply(adapter_name, axis=1)

results_df_clean.to_csv(os.path.join(RESULTS_DIR, "clean_results_for_report.csv"), index=False)

results_df_clean.head()

In [ ]:
summary_lines = [
    "Experiment summary",
    "==================",
    "",
]

for _, row in results_df_clean.iterrows():
    acc = row["accuracy"]
    loss = row["loss"]

    acc_str = f"{acc:.4f}" if pd.notna(acc) else "nan"
    loss_str = f"{loss:.4f}" if pd.notna(loss) else "nan"

    summary_lines.append(
        f"method={row['method']} | adapter_name={row['adapter_name']} | "
        f"step={row['step']} | eval_stage={row['eval_stage']} | "
        f"eval_set={row['eval_set']} | accuracy={acc_str} | loss={loss_str}"
    )

with open(os.path.join(RESULTS_DIR, "summary.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

In [ ]:
df_final = results_df_clean[
    (results_df_clean["eval_stage"] == "final") &
    (results_df_clean["step"] == num_steps)
].copy()

final_pivot = df_final.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

final_pivot = final_pivot[["first_step", "later_steps", "all_seen"]]

final_pivot["min_old_new"] = final_pivot[["first_step", "later_steps"]].min(axis=1)
final_pivot["harmonic_old_new"] = (
    2 * final_pivot["first_step"] * final_pivot["later_steps"]
    / (final_pivot["first_step"] + final_pivot["later_steps"] + 1e-12)
)

print("\nFINAL BALANCED METRICS")
print(final_pivot.sort_values("harmonic_old_new", ascending=False))

final_pivot.to_csv(os.path.join(RESULTS_DIR, "final_balanced_metrics.csv"))

In [ ]:
plot_df = final_pivot.reset_index()

methods = plot_df["method"].tolist()
metrics = ["first_step", "later_steps", "all_seen", "harmonic_old_new"]

x = np.arange(len(methods))
width = 0.18

plt.figure(figsize=(11, 5))

for i, metric in enumerate(metrics):
    plt.bar(x + (i - 1.5) * width, plot_df[metric], width, label=metric)

plt.xticks(x, methods, rotation=15)
plt.ylabel("Accuracy")
plt.title("Final comparison: RankExt vs baselines")
plt.legend()
plt.tight_layout()

out = os.path.join(PLOTS_DIR, "final_rankext_vs_baselines.png")
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", out)